# 03b – Sentiment & Issue-Type Labelling

**Methods:** VADER sentiment · Rule-based issue-type classification · Combined labelling  
**Outputs:** `sentiment_scores.csv` · `issue_type_labels.csv`


## 0. Setup

In [1]:
import os, re, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_ROOT   = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')


Setup complete.


## 1. Load Data

In [2]:
DATA_PATH = os.path.join(RESULTS_DIR, 'virtual_tickets.csv')
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found: {DATA_PATH}\n'
        'Run axis_2_topic_labeling.ipynb first.'
    )
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} tickets.')
df.head(3)


Loaded 200 tickets.


,ticket_id,text,true_topic,true_issue_type,true_sentiment,created_at
0,TKT-0001,I cannot log in to my account. The password re...,login,Account,negative,2024-05-20
1,TKT-0002,Error 500 is displayed whenever I submit the c...,bug,Bug,negative,2024-03-12
2,TKT-0003,Single sign-on with Google stopped working thi...,login,Account,negative,2024-10-06


## 2. VADER Sentiment Analysis

In [3]:
# ── Instantiate VADER ─────────────────────────────────────────────────────
analyser = SentimentIntensityAnalyzer()

def get_vader_scores(text: str) -> dict:
    """Return VADER polarity scores for a single text string."""
    return analyser.polarity_scores(text)

vader_records = df['text'].apply(get_vader_scores)
vader_df = pd.DataFrame(list(vader_records))
vader_df.columns = ['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']

# ── Map compound score → label ─────────────────────────────────────────────
def compound_to_label(score: float, pos_thresh=0.05, neg_thresh=-0.05) -> str:
    """Convert compound score to human-readable sentiment label.

    Parameters
    ----------
    score       : VADER compound score in [-1, 1]
    pos_thresh  : threshold above which sentiment is 'positive'
    neg_thresh  : threshold below which sentiment is 'negative'
    """
    if score >= pos_thresh:
        return 'positive'
    elif score <= neg_thresh:
        return 'negative'
    return 'neutral'

vader_df['vader_label'] = vader_df['vader_compound'].apply(compound_to_label)

df = pd.concat([df, vader_df], axis=1)
print('VADER score distribution:')
print(df['vader_label'].value_counts())


VADER score distribution:
vader_label
neutral     87
negative    64
positive    49
Name: count, dtype: int64


In [4]:
# ── Accuracy vs ground-truth ───────────────────────────────────────────────
correct = (df['vader_label'] == df['true_sentiment']).sum()
total   = len(df)
print(f'VADER accuracy (vs synthetic ground truth): {correct}/{total} = {correct/total:.2%}')


VADER accuracy (vs synthetic ground truth): 90/200 = 45.00%


## 3. Rule-Based Issue-Type Classifier

In [5]:
# ── Keyword rules (extend as needed) ──────────────────────────────────────
RULES = {
    'Bug': [
        r'crash', r'error\s*\d+', r'\berror\b', r'bug', r'fail',
        r'freeze', r'blank', r'not work', r'broken', r'timeout',
        r'duplicate', r'empty file', r'does not', r"doesn't",
    ],
    'Feature': [
        r'would.*great', r'please add', r'please allow', r'implement',
        r'feature', r'would love', r'need.*api', r'support for',
        r'can you', r'mobile app', r'keyboard shortcut',
    ],
    'Account': [
        r'log.*in', r'login', r'password', r'sign.*in', r'sign.*out',
        r'account', r'profile', r'email.*address', r'delete.*account',
        r'deactivat', r'ownership', r'gdpr',
    ],
}

# Compile patterns for efficiency
COMPILED_RULES = {
    label: [re.compile(p, re.IGNORECASE) for p in patterns]
    for label, patterns in RULES.items()
}

PRIORITY = ['Bug', 'Feature', 'Account']  # tie-breaking order

def classify_issue_type(text: str) -> str:
    """Assign an issue-type label via keyword matching.

    Returns 'Unknown' when no rule matches.
    """
    scores = {label: 0 for label in PRIORITY}
    for label, patterns in COMPILED_RULES.items():
        for pat in patterns:
            if pat.search(text):
                scores[label] += 1
    best_score = max(scores.values())
    if best_score == 0:
        return 'Unknown'
    for label in PRIORITY:
        if scores[label] == best_score:
            return label
    return 'Unknown'

df['rule_issue_type'] = df['text'].apply(classify_issue_type)
print('Rule-based issue-type distribution:')
print(df['rule_issue_type'].value_counts())


Rule-based issue-type distribution:
rule_issue_type
Account    71
Bug        56
Unknown    39
Feature    34
Name: count, dtype: int64


In [6]:
# ── Coverage & accuracy ───────────────────────────────────────────────────
labelled_mask = df['rule_issue_type'] != 'Unknown'
coverage  = labelled_mask.mean()
accuracy  = (df.loc[labelled_mask, 'rule_issue_type'] ==
             df.loc[labelled_mask, 'true_issue_type']).mean()

print(f'Label coverage : {coverage:.2%}')
print(f'Label accuracy : {accuracy:.2%}  (among covered tickets)')


Label coverage : 80.50%
Label accuracy : 90.68%  (among covered tickets)


## 4. Combined Sentiment × Issue-Type Labelling

In [7]:
df['combined_label'] = df['vader_label'] + '_' + df['rule_issue_type']
print('Combined label distribution:')
print(df['combined_label'].value_counts().head(15))


Combined label distribution:
combined_label
neutral_Account     57
negative_Bug        32
positive_Feature    25
negative_Unknown    22
positive_Bug        16
neutral_Unknown     13
negative_Account    10
neutral_Feature      9
neutral_Bug          8
positive_Account     4
positive_Unknown     4
Name: count, dtype: int64


## 5. Cross-Validation with Ground-Truth (Synthetic)

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

# ── Sentiment confusion matrix ─────────────────────────────────────────────
labels_sent = ['positive', 'neutral', 'negative']
cm_sent = confusion_matrix(df['true_sentiment'], df['vader_label'], labels=labels_sent)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_sent, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_sent, yticklabels=labels_sent, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('VADER Sentiment – Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'vader_sentiment_confusion.png'), dpi=100)
plt.close()
print('Sentiment confusion matrix saved.')

# ── Issue-type confusion matrix ───────────────────────────────────────────
labels_type = ['Bug', 'Feature', 'Account']
df_covered  = df[labelled_mask]
cm_type = confusion_matrix(
    df_covered['true_issue_type'], df_covered['rule_issue_type'],
    labels=labels_type,
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_type, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_type, yticklabels=labels_type, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Rule-based Issue Type – Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rule_issue_confusion.png'), dpi=100)
plt.close()
print('Issue-type confusion matrix saved.')

print('\nSentiment classification report:')
print(classification_report(df['true_sentiment'], df['vader_label'],
                             labels=labels_sent, zero_division=0))


Sentiment confusion matrix saved.
Issue-type confusion matrix saved.

Sentiment classification report:
              precision    recall  f1-score   support

    positive       0.41      0.80      0.54        25
     neutral       0.49      0.45      0.47        95
    negative       0.42      0.34      0.38        80

    accuracy                           0.45       200
   macro avg       0.44      0.53      0.46       200
weighted avg       0.45      0.45      0.44       200



## 6. Visualisations

In [9]:
# ── Sentiment distribution bar chart ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['vader_label'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#2196F3', '#FF9800', '#F44336'])
axes[0].set_title('VADER Predicted Sentiment')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['rule_issue_type'].value_counts().plot(
    kind='bar', ax=axes[1], color=['#4CAF50', '#9C27B0', '#FF5722', '#607D8B'])
axes[1].set_title('Rule-based Issue Types')
axes[1].set_xlabel('Issue Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Sentiment & Issue-Type Label Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sentiment_issuetype_distributions.png'), dpi=100)
plt.close()
print('Distribution plot saved.')


Distribution plot saved.


## 7. Save Outputs

In [10]:
# ── sentiment_scores.csv ─────────────────────────────────────────────────
sent_path = os.path.join(RESULTS_DIR, 'sentiment_scores.csv')
df[['ticket_id', 'text', 'true_sentiment',
    'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound',
    'vader_label']].to_csv(sent_path, index=False)
print(f'Saved: {sent_path}')

# ── issue_type_labels.csv ─────────────────────────────────────────────────
type_path = os.path.join(RESULTS_DIR, 'issue_type_labels.csv')
df[['ticket_id', 'text', 'true_issue_type', 'rule_issue_type', 'combined_label']].to_csv(
    type_path, index=False)
print(f'Saved: {type_path}')

print('\n03b complete.')


Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/sentiment_scores.csv
Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/issue_type_labels.csv

03b complete.
